# Getting started with `mitk.mxn.layout`

A tour of the typed Python DSL for MxN multi-widget layouts. Mirrored by `01_getting_started.py` -- keep both in sync when editing.

**Note:** this notebook is committed with executed cells so the `_repr_html_` output is visible when browsing the repo. Re-run the cells (Cell -> Run All) and commit before merging changes that touch the DSL surface. CI does not execute the notebook.

In [ ]:
from mitk.mxn.layout import (
    LayoutWindow,
    MxNLayoutDocument,
    Split,
    ViewDirection,
    grid,
    load_preset,
    save_preset,
)

## 1. Build a 2 x 3 grid with display labels

`grid(rows, cols, display_names=...)` returns a `Split`; wrap with `MxNLayoutDocument.create(...)` to get a complete document. Each leaf is auto-numbered as `mxn__widget0`, `mxn__widget1`, ... in pre-order.

In [ ]:
doc = MxNLayoutDocument.create(
    root=grid(
        2, 3,
        display_names=[
            "Tumor - Axial", "Tumor - Sagittal", "Tumor - Coronal",
            None, None, None,
        ],
    ),
    name="Tumor MPR + Reference",
)
doc

In [ ]:
doc.window_ids()

## 2. Compose a custom layout, then renumber ids cleanly

`Split.horizontal(...)` and `Split.vertical(...)` are the dataclass-builder pattern for arbitrary trees. After hand-stitching, `with_default_ids()` re-numbers leaves to the canonical pre-order convention.

**Identity warning:** ids are routing identities. Renaming an id that has already been served via REST or saved to a session breaks every cached reference. `with_default_ids()` is safe on freshly-built or freshly-loaded documents.

In [ ]:
custom = Split.vertical(
    Split.horizontal(
        LayoutWindow.create(id="mxn__handauthored.alpha", view_direction="axial"),
        LayoutWindow.create(id="mxn__handauthored.beta",  view_direction="sagittal"),
    ),
    LayoutWindow.create(id="mxn__handauthored.gamma", view_direction="coronal"),
)
custom_doc = MxNLayoutDocument.create(root=custom, name="Hand-stitched")
print("hand-authored ids:", custom_doc.window_ids())
renumbered = custom_doc.with_default_ids()
print("after with_default_ids():", renumbered.window_ids())
renumbered

## 3. Save and load a preset round-trip via a temp file

`save_preset` / `load_preset` write the canonical strict-mode v2.0 JSON. Round-trip equality holds for any in-memory document.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "preset.json"
    save_preset(path, doc)
    print(path.read_text(encoding="utf-8"))
    round_tripped = load_preset(path)

assert round_tripped == doc
round_tripped

## 4. Rename display labels via the selector

`MxNWindowSelector.with_display_name(...)` accepts a string, `None`, or a callable. The selector is immutable; the terminal returns a new document.

In [ ]:
relabelled = doc.select_windows.where(view_direction=ViewDirection.AXIAL).with_display_name(
    lambda w: f"Axial ({w.id.split('__', 1)[1]})"
)
for w in relabelled.windows():
    print(f"  {w.id}: {w.view_direction.value} -> name={w.name!r}")
relabelled

## 5. Re-group windows; auto-materialise the new group entry

`link_to(group)` on a document-rooted selector materialises a default `Group(name, select_all=True)` entry on the returned document if the name is new -- strict-mode invariants hold without manual bookkeeping.

In [ ]:
regrouped = doc.select_windows.where(view_direction="axial").link_to("row2")
print("groups after link_to('row2'):", sorted(regrouped.groups.keys()))
print("row2 members:", [w.id for w in regrouped.select_windows.where(group="row2")])
regrouped

## 6. Validation rejects malformed input at the cell boundary

Per-window structural rules are enforced once in `LayoutWindow.create` (the primary constructor). The bare dataclass init skips re-validation by design -- always go through `create()` for input from user-controlled sources.

In [ ]:
try:
    LayoutWindow.create(id="widget0", view_direction="axial")
except ValueError as exc:
    print("rejected as expected:")
    print(" ", exc)